<a href="https://colab.research.google.com/github/hangkimdiep-boop/RL_E1405_Submited_2026/blob/main/MICROGRID_ENERGY_OPTIMIZATION_PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MICROGRID ENERGY OPTIMIZATION - PPO (Proximal Policy Optimization)**

Author: Hang Diep Kim

Date: February 2026

**1️⃣ Install Dependencies & Imports**

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from collections import deque
import random
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import json, os, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")


🖥️ Using device: cuda


**2️⃣ PPO Configuration**



*   PPO HYPERPARAMETERS



In [ ]:
CONFIG = {
    # SETUP ENVIRONMENT
    "battery_capacity": 100,
    "battery_efficiency": 0.95,
    "max_charge_rate": 20,
    "max_discharge_rate": 20,
    "max_solar": 50,
    "max_wind": 30,
    "base_demand": 40,
    "demand_std": 10,
    "grid_price_min": 0.05,
    "grid_price_max": 0.25,
    "hours_per_episode": 24,

    # PPO SPECIFIC
    "state_dim": 8,
    "action_dim": 5,

    # NETWORK ARCHITECTURE
    "hidden_dims": [128, 128],

    # LEARNING RATE
    "lr_actor": 3e-4,
    "lr_critic": 1e-3,

    # PPO PARAMETERS
    "gamma": 0.99,               # Discount factor
    "gae_lambda": 0.95,          # GAE lambda
    "clip_epsilon": 0.2,         # PPO clipping range
    "ppo_epochs": 12,            # Update epochs per rollout
    "mini_batch_size": 64,       # Mini-batch size
    "entropy_coeff": 0.01,       # Entropy bonus
    "value_loss_coeff": 0.5,     # Value loss weight
    "max_grad_norm": 0.5,        # Gradient clipping

    # TRAINING
    "num_episodes": 1500,
    "max_steps_per_episode": 24, # 1 episode = 1 day (24h, step 1h)
    "rollout_steps": 96,         # Steps before update (4 episodes)
    "log_freq": 10,

    # RANDOM SEED
    "seed": 42,

    # REWARD
    "reward_renewable": 1.0,
    "reward_grid_penalty": -2.0,
    "reward_unmet_penalty": -5.0,
    "reward_battery_wear": -0.1,
    "reward_peak_bonus": 0.5,

    # TERMINATION
    "battery_critical_low": 0.05,
    "battery_critical_high": 1.0,
    "max_unmet_ratio": 0.20,
}

print("✅ PPO Config loaded!")
print(f"   LR Actor: {CONFIG['lr_actor']}, LR Critic: {CONFIG['lr_critic']}")
print(f"   Clip ε: {CONFIG['clip_epsilon']}, PPO Epochs: {CONFIG['ppo_epochs']}")
print(f"   GAE λ: {CONFIG['gae_lambda']}, Entropy: {CONFIG['entropy_coeff']}")